# Resampling of PEDAP

In [56]:
import os
import numpy as np
import pandas as pd
from datetime import datetime

### Read Existing Tables

In [57]:
file_path = "../../data/out/PEDAP Public Dataset - Release 3 - 2024-09-25/"
file_map = {
    'cgm': 'PEDAP_cgm_history.csv.gz',
    'bolus': 'PEDAP_bolus_event_history.csv.gz',
    'basal': 'PEDAP_basal_event_history.csv.gz',
}

In [58]:
def get_df_from_file(file_path, file_name, parse_datetime=True, sep=','):
    df = pd.read_csv(file_path + file_name, sep=sep)
    if parse_datetime:
        df['date'] = pd.to_datetime(df['datetime'], unit='s')
    return df

In [59]:
def get_extended_df(original_df, value_column):
    """
    Get a df where quantities are distributed throughout 5-minute intervals instead of having start- and end dates.
    """
    new_rows = []
    for _, row in original_df.iterrows():
        new_rows.extend(split_duration(row, value_column))
    extended_df = pd.DataFrame(new_rows)
    extended_df.set_index('date', inplace=True)
    return extended_df

def split_duration(row, value_column):
    """
    For features with a duration, we split the values across 5-minute intervals by adding
    new rows for every 5-minute window in duration, and equally split the original quantity across those rows.
    """
    duration = row['end_date'] - row['date']
    rounded_duration = round(duration / pd.Timedelta(minutes=5)) * pd.Timedelta(minutes=5)
    num_intervals = rounded_duration // pd.Timedelta(minutes=5)
    if num_intervals < 1:
        num_intervals = 1
    value_per_interval = row[value_column] / num_intervals
    new_rows = []
    for i in range(int(num_intervals)):
        new_row = {
            'date': row['date'] + pd.Timedelta(minutes=5 * i),
            value_column: value_per_interval,
            'patient_id': row['patient_id'],
        }
        new_rows.append(new_row)
    return new_rows

In [60]:
def parse_flair_dates(dates, format_date = '%m/%d/%Y', format_time = '%I:%M:%S %p'):
    """Parse date strings separately for those with/without time component, interpret those without as midnight (00AM)
    Args:
        dates (pd.DataFrame): datetimes (string) either in in the %m/%d/%Y or %m/%d/%Y %I:%M:%S %p format
    Returns:
        pandas series: with parsed dates
    """
    #make sure to only parse dates if the value is not null
    dates = dates.astype(str)
    only_date = dates.apply(len) <=10
    dates_copy = dates.copy()
    dates_copy.loc[only_date] = pd.to_datetime(dates.loc[only_date], format=format_date)
    dates_copy.loc[~only_date] = pd.to_datetime(dates.loc[~only_date], format=f'{format_date} {format_time}')
    return dates_copy.astype('datetime64[ns]')

In [61]:
df_glucose = get_df_from_file(file_path, file_map['cgm'])
df_bolus = get_df_from_file(file_path, file_map['bolus'])
df_basal = get_df_from_file(file_path, file_map['basal'])


### Resample Existing Tables

In [62]:
df_glucose.set_index('date', inplace=True)
df_glucose.head()

,patient_id,datetime,cgm
date,,,
2020-12-28 14:17:16,27,1609165036,83
2020-12-28 14:22:16,27,1609165336,104
2020-12-28 14:27:16,27,1609165636,99
2020-12-28 14:32:16,27,1609165936,95
2020-12-28 14:37:17,27,1609166237,84


In [63]:
df_bolus_orig = df_bolus.copy()
df_bolus['end_date'] = df_bolus['date'] + pd.to_timedelta(df_bolus['delivery_duration'], unit='s')
df_bolus = get_extended_df(df_bolus, 'bolus')
df_bolus

,bolus,patient_id
date,,
2020-12-28 13:28:01,1.1100,27
2020-12-28 14:41:32,0.0800,27
2020-12-28 16:22:16,0.2611,27
2020-12-28 16:25:40,0.0900,27
2020-12-28 17:28:41,0.1300,27
...,...,...
2022-03-23 12:23:49,0.0710,38
2022-03-24 06:47:40,0.0710,38
2022-03-24 06:52:40,0.0710,38


In [64]:
print(f'New sum after distribution of extended boluses: {df_bolus["bolus"].sum():.2f}, should be: {df_bolus_orig["bolus"].sum():.2f}')

New sum after distribution of extended boluses: 170207.49, should be: 170207.49


In [65]:
df_basal_orig = df_basal.copy()
df_basal.sort_values(by=['patient_id', 'date'], inplace=True)
df_basal.set_index('date', inplace=True)
df_basal

,patient_id,basal_rate,datetime
date,,,
2021-09-28 11:59:13,3,0.150,1632830353
2021-09-28 12:01:25,3,0.000,1632830485
2021-09-28 12:05:22,3,0.556,1632830722
2021-09-28 12:10:22,3,0.150,1632831022
2021-09-28 12:38:32,3,0.000,1632832712
...,...,...,...
2021-11-08 16:00:06,109,0.300,1636387206
2021-11-08 16:05:07,109,0.300,1636387507
2021-11-08 16:10:08,109,0.300,1636387808


In [66]:
processed_dfs = []
subject_ids = df_glucose['patient_id'].unique()
print("Subjects:", len(subject_ids))
for subject_id in subject_ids:
    df_subject = df_glucose[df_glucose['patient_id'] == subject_id].copy()
    df_subject = df_subject[['cgm']].resample('5min', label='right').mean()
    df_subject['patient_id'] = subject_id
    df_subject.sort_index(inplace=True)

    def merge_data(df_col, df_subject, col_names, subject_id, agg_type='sum'):
        """ agg_type is data aggregation type. """
        df_subset = df_col[df_col['patient_id'] == subject_id].copy()
        if not df_subset.empty:
            if agg_type == 'mean':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').mean()
            elif agg_type == 'sum':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').sum()
            elif agg_type == 'first':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').first()
            elif agg_type == 'last':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').last()
            else:
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').sum()
            df_subject = pd.merge(df_subject, df_subset, on="date", how='outer')
        else:
            df_subject[col_names] = np.nan
        return df_subject

    # Add insulin and insulin type
    df_subject = merge_data(df_bolus, df_subject, ['bolus'], subject_id, agg_type='sum')
    df_subject = merge_data(df_basal, df_subject, ['basal_rate'], subject_id, agg_type='last')
    df_subject['basal_rate'] = df_subject['basal_rate'].ffill()
    
    df_subject['patient_id'] = subject_id
    df_subject = df_subject.rename(columns={'patient_id': 'id', 'basal_rate': 'basal', 'cgm': 'CGM'})
    processed_dfs.append(df_subject)
    print(f"{subject_id} is finished processing")

df_final = pd.concat(processed_dfs)

Subjects: 65
27 is finished processing
8 is finished processing
93 is finished processing
103 is finished processing
4 is finished processing
26 is finished processing
77 is finished processing
82 is finished processing
59 is finished processing
31 is finished processing
44 is finished processing
72 is finished processing
9 is finished processing
13 is finished processing
38 is finished processing
87 is finished processing
81 is finished processing
14 is finished processing
66 is finished processing
19 is finished processing
83 is finished processing
51 is finished processing
36 is finished processing
24 is finished processing
58 is finished processing
84 is finished processing
20 is finished processing
95 is finished processing
90 is finished processing
91 is finished processing
53 is finished processing
16 is finished processing
70 is finished processing
88 is finished processing
64 is finished processing
60 is finished processing
10 is finished processing
23 is finished processing
6

In [67]:
df_final

,CGM,id,bolus,basal
date,,,,
2020-12-28 13:30:00,NaN,27,1.11,0.3
2020-12-28 13:35:00,NaN,27,0.00,0.3
2020-12-28 13:40:00,NaN,27,0.00,0.3
2020-12-28 13:45:00,NaN,27,0.00,0.3
2020-12-28 13:50:00,NaN,27,0.00,0.3
...,...,...,...,...
2021-12-12 19:00:00,126.0,89,NaN,0.0
2021-12-12 19:05:00,125.0,89,NaN,0.0
2021-12-12 19:10:00,128.0,89,NaN,0.0


### Add Additional Tables

We found:
- Carbs
- Insulin type
- Age
- Weight
- Height
- Gender

In [68]:
!pip install striprtf


[notice] A new release of pip available: 22.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [69]:
data_glossary = "../../data/raw/PEDAP Public Dataset - Release 3 - 2024-09-25/DataGlossary.rtf"
from striprtf.striprtf import rtf_to_text
with open(data_glossary, "r", encoding="utf-8") as file:
    rtf_content = file.read()

plain_text = rtf_to_text(rtf_content)
print(plain_text)

PEDAP Protocol – Public Dataset

Link to public website:  https://public.jaeb.org/datasets/diabetes
Protocol details: see full protocol PDF, included with this dataset
Manuscripts:
Primary: https://www.nejm.org/doi/full/10.1056/NEJMoa2210834
Patient-Reported Outcomes (PROs): https://pubmed.ncbi.nlm.nih.gov/38278493/
Table of Contents – Data Files, sorted by Source, then Data File Name

Data Table List:
Data from Study Devices:|
PEDAPDexcomClarityCalibration|CGM Calibration Data from Dexcom Clarity|
PEDAPDexcomClarityCGM|CGM Data from Dexcom Clarity|
PEDAPKetone|Ketone Data from Precision Xtra meter|
PEDAPOtherCGM|CGM Data from sources other than Dexcom Clarity|
PEDAPTandemBASALRATECHG|Event logged on pump when insulin basal rate changes due to pumping events|
PEDAPTandemBOLUSDELIVERED|Event logged on pump when delivery of an insulin bolus (Standard, Extended, or Automatic) is completed|
PEDAPTandemCGMDataGXB|Event logged on pump when a CGM value is obtained from a connected Dexcom CGM 

In [70]:
raw_data_file_path = "../../data/raw/PEDAP Public Dataset - Release 3 - 2024-09-25/Data Files/"

In [71]:
# Adding carbs
df_carbs = get_df_from_file(raw_data_file_path, 'PEDAPTandemBolusDelivered.txt', parse_datetime=False, sep='|')
df_carbs.head()

,PtID,RecID,BolusID,DeviceDtTm,BolusAmount,CarbAmount,BolusType,Duration,ExtendedBolusPortion
0,27,1.0,2,12/28/2020 1:28:01 PM,1.110000,22,Standard,0,NaN
1,27,2.0,3,12/28/2020 2:41:32 PM,0.080000,2,Standard,0,NaN
2,27,3.0,4,12/28/2020 4:22:16 PM,0.261078,0,Automatic,0,NaN
3,27,4.0,5,12/28/2020 4:25:40 PM,0.090000,0,Standard,0,NaN
4,27,5.0,7,12/28/2020 5:28:41 PM,0.130000,12,Standard,0,NaN


In [72]:
df_carbs[['CarbAmount', 'DeviceDtTm', 'PtID']][df_carbs['PtID'] == 3].sort_values(by=['PtID', 'DeviceDtTm'])

,CarbAmount,DeviceDtTm,PtID
106715,40,1/1/2022 10:17:58 AM,3
106716,0,1/1/2022 11:55:55 AM,3
106717,40,1/1/2022 3:09:42 PM,3
106718,0,1/1/2022 4:55:18 PM,3
106719,0,1/1/2022 6:35:18 PM,3
...,...,...,...
119068,0,9/30/2021 3:17:43 PM,3
119069,0,9/30/2021 5:15:19 PM,3
119070,39,9/30/2021 7:16:31 PM,3
69798,0,9/30/2021 8:56:20 PM,3


In [73]:
# Comparing this table to the second carbs table
df_meal_exercise_log = get_df_from_file(raw_data_file_path, 'PEDAPMealExerciseLog.txt', parse_datetime=False, sep='|')
df_meal_exercise_log#[['MealCarbs', 'SessionDt', 'PtID', 'SessionType']].sort_values(by=['PtID', 'SessionDt'])

,RecID,PtID,ParentLoginVisitID,ParentLoginID,SessionType,SessionDt,MealStartHr,MealStartMin,MealStartAMPM,MealEndHr,MealEndMin,MealEndAMPM,MealCarbs,MealCarbsUnknown,ExerciseStartHr,ExerciseStartMin,ExerciseStartAMPM,ExerciseEndHr,ExerciseEndMin,ExerciseEndAMPM
0,1,84,NaN,6405,1 - Skip Meal Bolus,7/17/2021,12,56,PM,1,27,PM,26,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,84,NaN,6494,2 - Full Meal Bolus Plus Exercise,7/22/2021,1,21,PM,1,51,PM,54,NaN,2.0,34.0,PM,3.0,49.0,PM
2,3,44,NaN,6941,1 - Skip Meal Bolus,7/16/2021,12,0,PM,12,20,PM,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,44,NaN,6946,3 - Exercise,7/14/2021,11,55,AM,12,10,PM,36,NaN,1.0,40.0,PM,2.0,10.0,PM
4,5,30,NaN,6951,1 - Skip Meal Bolus,3/9/2021,5,15,PM,5,30,PM,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,102,101,NaN,9897,1 - Skip Meal Bolus,9/1/2021,5,10,PM,5,30,PM,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
101,103,101,NaN,9898,2 - Full Meal Bolus Plus Exercise,9/7/2021,3,55,PM,4,5,PM,25,NaN,4.0,35.0,PM,5.0,5.0,PM
102,104,101,NaN,9899,3 - Exercise,9/9/2021,11,30,AM,11,40,AM,30,NaN,1.0,30.0,PM,2.0,0.0,PM
103,105,21,NaN,10136,1 - Skip Meal Bolus,7/18/2021,3,15,PM,3,25,PM,27,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [74]:
df_meal_exercise_log[df_meal_exercise_log['MealCarbs'] > 0].shape

(105, 20)

In [75]:
df_meal_exercise_log[df_meal_exercise_log['ExerciseStartMin'].notna()].shape

(68, 20)

In [76]:
df_carbs[df_carbs['CarbAmount'] > 0].shape

(135985, 9)

In [77]:
# Compared to the exercise / sleep mode table
df_exercise = get_df_from_file(raw_data_file_path, 'PEDAPTandemUserModeChange.txt', parse_datetime=False, sep='|')
df_exercise.sort_values(by=['PtID', 'DeviceDtTm'])#['CurrentMode'] == 'Exercise']

,PtID,RecID,DeviceDtTm,CurrentMode,PreviousMode
12585,1,12610,10/1/2021 10:00:42 PM,Sleep,Normal
12584,1,12609,10/1/2021 9:31:42 AM,Normal,Sleep
12603,1,12628,10/10/2021 10:00:42 PM,Sleep,Normal
12602,1,12627,10/10/2021 9:31:42 AM,Normal,Sleep
12605,1,12630,10/11/2021 10:00:43 PM,Sleep,Normal
...,...,...,...,...,...
22890,109,22993,9/7/2021 7:30:14 PM,Sleep,Normal
22891,109,22994,9/8/2021 6:31:13 AM,Normal,Sleep
22892,109,22995,9/8/2021 7:30:14 PM,Sleep,Normal
22894,109,22997,9/9/2021 12:32:54 PM,Sleep,Normal


We observe that the modes data is looking strange, there is no consistency in exercise events with subsequent non exericse events, and there seem to be duplicates as well. So this data is not usable for storing exercise events. 

Hence, we use the carbs df for carbs, and the log df for the 68 exercise events.

In [78]:
# Format df carbs
# Adding carbs
df_carbs = get_df_from_file(raw_data_file_path, 'PEDAPTandemBolusDelivered.txt', parse_datetime=False, sep='|')
df_carbs['date'] = parse_flair_dates(df_carbs['DeviceDtTm'])
df_carbs = df_carbs[df_carbs['CarbAmount'] > 0][['PtID', 'date', 'CarbAmount']]
df_carbs = df_carbs.rename(columns={'PtID': 'id', 'CarbAmount': 'carbs'})
df_carbs.head()

,id,date,carbs
0,27,2020-12-28 13:28:01,22
1,27,2020-12-28 14:41:32,2
4,27,2020-12-28 17:28:41,12
6,27,2020-12-29 06:03:20,14
9,27,2020-12-29 09:55:56,17


In [79]:
# Add carbs to df final
processed_dfs = []
subject_ids = df_final['id'].unique()
print("Subjects:", len(subject_ids))
for subject_id in subject_ids:
    df_subject = df_final[df_final['id'] == subject_id].copy()
    df_subject.sort_index(inplace=True)

    # Add carbs
    df_subject_carbs = df_carbs[df_carbs['id'] == subject_id].copy()
    df_subject_carbs.set_index('date', inplace=True)
    df_subject_carbs = df_subject_carbs[df_subject_carbs['carbs'].notna()]['carbs'].resample('5min', label='right').sum()
    df_subject = pd.merge(df_subject, df_subject_carbs, on="date", how='outer')
    
    df_subject['id'] = subject_id
    processed_dfs.append(df_subject)
    print(f"{subject_id} is finished processing")

df_final = pd.concat(processed_dfs)
df_final

Subjects: 65
27 is finished processing
8 is finished processing
93 is finished processing
103 is finished processing
4 is finished processing
26 is finished processing
77 is finished processing
82 is finished processing
59 is finished processing
31 is finished processing
44 is finished processing
72 is finished processing
9 is finished processing
13 is finished processing
38 is finished processing
87 is finished processing
81 is finished processing
14 is finished processing
66 is finished processing
19 is finished processing
83 is finished processing
51 is finished processing
36 is finished processing
24 is finished processing
58 is finished processing
84 is finished processing
20 is finished processing
95 is finished processing
90 is finished processing
91 is finished processing
53 is finished processing
16 is finished processing
70 is finished processing
88 is finished processing
64 is finished processing
60 is finished processing
10 is finished processing
23 is finished processing
6

,CGM,id,bolus,basal,carbs
date,,,,,
2020-12-28 13:30:00,NaN,27,1.11,0.3,22.0
2020-12-28 13:35:00,NaN,27,0.00,0.3,0.0
2020-12-28 13:40:00,NaN,27,0.00,0.3,0.0
2020-12-28 13:45:00,NaN,27,0.00,0.3,0.0
2020-12-28 13:50:00,NaN,27,0.00,0.3,0.0
...,...,...,...,...,...
2021-12-12 19:00:00,126.0,89,NaN,0.0,NaN
2021-12-12 19:05:00,125.0,89,NaN,0.0,NaN
2021-12-12 19:10:00,128.0,89,NaN,0.0,NaN


In [97]:
# Adding exercise label and duration
df_exercise = get_df_from_file(raw_data_file_path, 'PEDAPMealExerciseLog.txt', parse_datetime=False, sep='|')
df_exercise = df_exercise[df_exercise['ExerciseStartMin'].notna()]

def get_datetime_from_separate_cols(df, date_col, hour_col, min_col, ampm_col):
    date_str = df[date_col]
    hour_str = df[hour_col].astype(int).astype(str)
    min_str = df[min_col].astype(int).astype(str).str.zfill(2)
    datetime_str = date_str + ' ' + hour_str + ':' + min_str + ' ' + df[ampm_col]
    return pd.to_datetime(datetime_str, format='%m/%d/%Y %I:%M %p')

df_exercise['date'] = get_datetime_from_separate_cols(df_exercise, 'SessionDt', 'ExerciseStartHr', 'ExerciseStartMin', 'ExerciseStartAMPM')
df_exercise['end_date'] = get_datetime_from_separate_cols(df_exercise, 'SessionDt', 'ExerciseEndHr', 'ExerciseEndMin', 'ExerciseEndAMPM')
df_exercise['workout_duration'] = ((df_exercise['end_date'] - df_exercise['date']).dt.total_seconds() / 60).astype(int)
df_exercise = df_exercise[['PtID', 'date', 'workout_duration']]
df_exercise = df_exercise.rename(columns={'PtID': 'id'})
df_exercise['workout_label'] = 'Exercise'
df_exercise

,id,date,workout_duration,workout_label
1,84,2021-07-22 14:34:00,75,Exercise
3,44,2021-07-14 13:40:00,30,Exercise
5,30,2021-03-11 18:10:00,30,Exercise
6,30,2021-03-21 13:54:00,32,Exercise
7,70,2021-08-21 14:20:00,44,Exercise
...,...,...,...,...
98,33,2021-06-22 12:15:00,40,Exercise
99,33,2021-06-30 14:10:00,30,Exercise
101,101,2021-09-07 16:35:00,30,Exercise
102,101,2021-09-09 13:30:00,30,Exercise


In [98]:
# Add workout label and duration to the df_final
processed_dfs = []
subject_ids = df_final['id'].unique()
print("Subjects:", len(subject_ids))
for subject_id in subject_ids:
    df_subject = df_final[df_final['id'] == subject_id].copy()
    df_subject.sort_index(inplace=True)

    df_subject_workouts = df_exercise[df_exercise['id'] == subject_id].copy()
    df_subject_workouts.set_index('date', inplace=True)
    df_subject_workouts = df_subject_workouts[['workout_label', 'workout_duration']].resample('5min', label='right').last()
    df_subject = pd.merge(df_subject, df_subject_workouts, on="date", how='outer')
    
    df_subject['id'] = subject_id
    processed_dfs.append(df_subject)
    print(f"{subject_id} is finished processing")

df_final = pd.concat(processed_dfs)
df_final

Subjects: 65
27 is finished processing
8 is finished processing
93 is finished processing
103 is finished processing
4 is finished processing
26 is finished processing
77 is finished processing
82 is finished processing
59 is finished processing
31 is finished processing
44 is finished processing
72 is finished processing
9 is finished processing
13 is finished processing
38 is finished processing
87 is finished processing
81 is finished processing
14 is finished processing
66 is finished processing
19 is finished processing
83 is finished processing
51 is finished processing
36 is finished processing
24 is finished processing
58 is finished processing
84 is finished processing
20 is finished processing
95 is finished processing
90 is finished processing
91 is finished processing
53 is finished processing
16 is finished processing
70 is finished processing
88 is finished processing
64 is finished processing
60 is finished processing
10 is finished processing
23 is finished processing
6

,CGM,id,bolus,basal,carbs,workout_label,workout_duration
date,,,,,,,
2020-12-28 13:30:00,NaN,27,1.11,0.3,22.0,NaN,NaN
2020-12-28 13:35:00,NaN,27,0.00,0.3,0.0,NaN,NaN
2020-12-28 13:40:00,NaN,27,0.00,0.3,0.0,NaN,NaN
2020-12-28 13:45:00,NaN,27,0.00,0.3,0.0,NaN,NaN
2020-12-28 13:50:00,NaN,27,0.00,0.3,0.0,NaN,NaN
...,...,...,...,...,...,...,...
2021-12-12 19:00:00,126.0,89,NaN,0.0,NaN,NaN,NaN
2021-12-12 19:05:00,125.0,89,NaN,0.0,NaN,NaN,NaN
2021-12-12 19:10:00,128.0,89,NaN,0.0,NaN,NaN,NaN


In [99]:
df_final[df_final['workout_label'].notna()]

,CGM,id,bolus,basal,carbs,workout_label,workout_duration
date,,,,,,,
2021-07-14 13:45:00,164.0,44,0.000000,0.000,0.0,Exercise,30.0
2021-04-26 17:00:00,98.0,36,0.000000,0.000,0.0,Exercise,30.0
2021-05-08 16:40:00,298.0,36,0.085208,0.000,0.0,Exercise,30.0
2021-07-22 14:35:00,249.0,84,0.000000,0.000,0.0,Exercise,75.0
2021-08-07 15:25:00,210.0,84,0.000000,0.000,0.0,Exercise,36.0
2021-11-04 18:40:00,132.0,95,0.000000,0.320,0.0,Exercise,30.0
2021-11-12 10:45:00,164.0,95,0.000000,0.320,0.0,Exercise,106.0
2021-03-02 13:00:00,226.0,91,0.000000,0.000,0.0,Exercise,30.0
2021-03-09 14:55:00,250.0,91,0.000000,0.000,0.0,Exercise,30.0


In [101]:
df_insulin_type = get_df_from_file(raw_data_file_path, 'PEDAPInsulin.txt', parse_datetime=False, sep='|')
df_insulin_type = df_insulin_type[['PtID', 'InsulinName']]
df_insulin_type = df_insulin_type.rename(columns={'PtID': 'id', 'InsulinName': 'insulin_type'})
df_insulin_type

,id,insulin_type
0,93,Novolog Fiasp
1,27,Humalog (Lispro)
2,26,Novolog (Aspart)
3,93,Novolog (Aspart)
4,50,Humalog (Lispro)
...,...,...
188,49,Humalog (Lispro)
189,49,"Humalog (Lispro, U100)"
190,49,"Humalog (Lispro, U100)"
191,90,"Humalog (Lispro, U100)"


In [102]:
def add_single_value_to_subjects(df, df_new_val, col_name):
    # TODO: this function is very inefficient... add value directly to located rows instead
    processed_dfs = []
    subject_ids = df['id'].unique()
    for subject_id in subject_ids:
        df_subject = df[df['id'] == subject_id].copy()
        df_subject.sort_index(inplace=True)
    
        user_data = df_new_val[df_new_val['id'] == subject_id].copy()
        if not user_data.empty:
            df_subject[col_name] = user_data[col_name].iloc[0]
        else:
            df_subject[col_name] = np.nan        
        processed_dfs.append(df_subject)
        
    df = pd.concat(processed_dfs)
    return df

In [103]:
# Add insulin type
df_final = add_single_value_to_subjects(df_final, df_insulin_type, 'insulin_type')
df_final

,CGM,id,bolus,basal,carbs,workout_label,workout_duration,insulin_type
date,,,,,,,,
2020-12-28 13:30:00,NaN,27,1.11,0.3,22.0,NaN,NaN,Humalog (Lispro)
2020-12-28 13:35:00,NaN,27,0.00,0.3,0.0,NaN,NaN,Humalog (Lispro)
2020-12-28 13:40:00,NaN,27,0.00,0.3,0.0,NaN,NaN,Humalog (Lispro)
2020-12-28 13:45:00,NaN,27,0.00,0.3,0.0,NaN,NaN,Humalog (Lispro)
2020-12-28 13:50:00,NaN,27,0.00,0.3,0.0,NaN,NaN,Humalog (Lispro)
...,...,...,...,...,...,...,...,...
2021-12-12 19:00:00,126.0,89,NaN,0.0,NaN,NaN,NaN,Humalog (Lispro)
2021-12-12 19:05:00,125.0,89,NaN,0.0,NaN,NaN,NaN,Humalog (Lispro)
2021-12-12 19:10:00,128.0,89,NaN,0.0,NaN,NaN,NaN,Humalog (Lispro)


In [107]:
# Add age
df_age = get_df_from_file(raw_data_file_path, 'PtRoster.txt', parse_datetime=False, sep='|')
df_age = df_age[['PtID', 'AgeAsofEnrollDt']]
df_age = df_age.rename(columns={'PtID': 'id', 'AgeAsofEnrollDt': 'age'})
df_age

,id,age
0,93,2
1,26,4
2,57,4
3,82,2
4,44,3
...,...,...
104,69,4
105,76,3
106,5,2
107,43,3


In [108]:
# Add age
df_final = add_single_value_to_subjects(df_final, df_age, 'age')
df_final.head()

,CGM,id,bolus,basal,carbs,workout_label,workout_duration,insulin_type,age
date,,,,,,,,,
2020-12-28 13:30:00,NaN,27,1.11,0.3,22.0,NaN,NaN,Humalog (Lispro),4
2020-12-28 13:35:00,NaN,27,0.00,0.3,0.0,NaN,NaN,Humalog (Lispro),4
2020-12-28 13:40:00,NaN,27,0.00,0.3,0.0,NaN,NaN,Humalog (Lispro),4
2020-12-28 13:45:00,NaN,27,0.00,0.3,0.0,NaN,NaN,Humalog (Lispro),4
2020-12-28 13:50:00,NaN,27,0.00,0.3,0.0,NaN,NaN,Humalog (Lispro),4


In [115]:
# Add gender
df_gender = get_df_from_file(raw_data_file_path, 'PEDAPDiabScreening.txt', parse_datetime=False, sep='|')
df_gender = df_gender[['PtID', 'Sex']]
df_gender = df_gender.rename(columns={'PtID': 'id', 'Sex': 'gender'})
df_gender

,id,gender
0,93,M
1,27,F
2,26,M
3,50,F
4,8,F
...,...,...
100,94,M
101,55,M
102,101,M
103,71,M


In [116]:
# Add weight, height, and gender
df_final = add_single_value_to_subjects(df_final, df_gender, 'gender')
df_final.head()

,CGM,id,bolus,basal,carbs,workout_label,workout_duration,insulin_type,age,gender
date,,,,,,,,,,
2020-12-28 13:30:00,NaN,27,1.11,0.3,22.0,NaN,NaN,Humalog (Lispro),4,F
2020-12-28 13:35:00,NaN,27,0.00,0.3,0.0,NaN,NaN,Humalog (Lispro),4,F
2020-12-28 13:40:00,NaN,27,0.00,0.3,0.0,NaN,NaN,Humalog (Lispro),4,F
2020-12-28 13:45:00,NaN,27,0.00,0.3,0.0,NaN,NaN,Humalog (Lispro),4,F
2020-12-28 13:50:00,NaN,27,0.00,0.3,0.0,NaN,NaN,Humalog (Lispro),4,F


In [125]:
# Add weight and height
df_weight_height = get_df_from_file(raw_data_file_path, 'PEDAPDiabPhysExam.txt', parse_datetime=False, sep='|')

print(df_weight_height['HeightUnits'].unique())
print(df_weight_height['WeightUnits'].unique())

# If inches or lbs, --> cm and kg
df_weight_height['Height'] = df_weight_height.apply(
    lambda row: row['Height'] * 2.54 if row['HeightUnits'] != 'cm' else row['Height'],
    axis=1
)
df_weight_height['Weight'] = df_weight_height.apply(
    lambda row: row['Weight'] * 0.453592 if row['WeightUnits'] != 'kg' else row['Weight'],
    axis=1
)
df_weight_height[['Weight', 'Height']] = df_weight_height[['Weight', 'Height']].round(1)
df_weight_height = df_weight_height[['PtID', 'Weight', 'Height']]
df_weight_height = df_weight_height.rename(columns={'PtID': 'id', 'Weight': 'weight', 'Height': 'height'})
df_weight_height

['in' 'cm' nan]
['lbs' 'kg']


,id,weight,height
0,93,15.4,94.0
1,27,19.6,104.4
2,26,21.3,109.2
3,50,18.1,96.5
4,8,17.9,106.7
...,...,...,...
100,94,14.1,96.5
101,55,14.0,96.5
102,101,13.6,91.4
103,71,20.8,112.0


In [126]:
# Add weight, height, and gender
df_final = add_single_value_to_subjects(df_final, df_weight_height, 'weight')
df_final = add_single_value_to_subjects(df_final, df_weight_height, 'height')
df_final.head()

,CGM,id,bolus,basal,carbs,workout_label,workout_duration,insulin_type,age,gender,weight,height
date,,,,,,,,,,,,
2020-12-28 13:30:00,NaN,27,1.11,0.3,22.0,NaN,NaN,Humalog (Lispro),4,F,19.6,104.4
2020-12-28 13:35:00,NaN,27,0.00,0.3,0.0,NaN,NaN,Humalog (Lispro),4,F,19.6,104.4
2020-12-28 13:40:00,NaN,27,0.00,0.3,0.0,NaN,NaN,Humalog (Lispro),4,F,19.6,104.4
2020-12-28 13:45:00,NaN,27,0.00,0.3,0.0,NaN,NaN,Humalog (Lispro),4,F,19.6,104.4
2020-12-28 13:50:00,NaN,27,0.00,0.3,0.0,NaN,NaN,Humalog (Lispro),4,F,19.6,104.4


### Save Resampled Data

In [127]:
df_final

,CGM,id,bolus,basal,carbs,workout_label,workout_duration,insulin_type,age,gender,weight,height
date,,,,,,,,,,,,
2020-12-28 13:30:00,NaN,27,1.11,0.3,22.0,NaN,NaN,Humalog (Lispro),4,F,19.6,104.4
2020-12-28 13:35:00,NaN,27,0.00,0.3,0.0,NaN,NaN,Humalog (Lispro),4,F,19.6,104.4
2020-12-28 13:40:00,NaN,27,0.00,0.3,0.0,NaN,NaN,Humalog (Lispro),4,F,19.6,104.4
2020-12-28 13:45:00,NaN,27,0.00,0.3,0.0,NaN,NaN,Humalog (Lispro),4,F,19.6,104.4
2020-12-28 13:50:00,NaN,27,0.00,0.3,0.0,NaN,NaN,Humalog (Lispro),4,F,19.6,104.4
...,...,...,...,...,...,...,...,...,...,...,...,...
2021-12-12 19:00:00,126.0,89,NaN,0.0,NaN,NaN,NaN,Humalog (Lispro),2,M,13.3,88.9
2021-12-12 19:05:00,125.0,89,NaN,0.0,NaN,NaN,NaN,Humalog (Lispro),2,M,13.3,88.9
2021-12-12 19:10:00,128.0,89,NaN,0.0,NaN,NaN,NaN,Humalog (Lispro),2,M,13.3,88.9


In [128]:
save_file_path = "../../data/resampled/"
os.makedirs(save_file_path, exist_ok=True)
df_final.to_csv(save_file_path + 'PEDAP.csv')